# Day 2 — The Data Is Never Clean

**Theme:** Every real dataset has dirty values, missing fields, and columns named by someone who didn't think about you.
Today you will feel that pain — and build the tools to fix it.

**By the end of this notebook you will have:**
- Loaded a raw 311 CSV into a pandas DataFrame
- Explored its shape, column names, and null counts
- Written three cleaning functions from scratch:
  - `clean_complaint_type` — normalize the messy free-text complaint category
  - `parse_created_date` — convert the date string into a real Python datetime
  - `flag_missing_location` — mark rows where lat/lon is absent
- Applied those functions to the full DataFrame

**Ground rules for today:**
You write every line of code. Errors are information, not failures — read the error message before asking for help.
If you get stuck for a while, the next cell will have a hint.

## Step 1 — Get the data

Go to the **NYC Open Data portal** (`data.cityofnewyork.us`) and search for **"311 Service Requests"**.
You want the dataset called *311 Service Requests from 2010 to Present*.

That dataset has millions of rows. You do **not** want all of them. Use the site's filter tool to:
1. Filter `Borough` = **BROOKLYN**
2. Filter `Created Date` to a recent 1-month window (e.g. March 2024)

Then export as CSV and save it to your `data/` folder. Call it something like `data/311_brooklyn_march2024.csv`.

**Come back here once the file is in `data/`.**

In [43]:
import pandas as pd
df = '../data/311_Service_Requests_from_2020_to_Present_20260608.csv'
pd.read_csv(df) 


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,69189924,05/31/2026 11:44:40 PM,06/01/2026 02:08:25 AM,NYPD,New York City Police Department,Illegal Parking,Commercial Overnight Parking,NaN,Street/Sidewalk,11214.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.612992,-74.002993,POINT (-74.00299305542 40.612992270116)
1,69195629,05/31/2026 11:44:28 PM,06/01/2026 02:08:47 AM,NYPD,New York City Police Department,Non-Emergency Police Matter,Trespassing,NaN,Residential Building/House,11212.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.661356,-73.907126,POINT (-73.907126367225 40.66135638569)
2,69188434,05/31/2026 11:43:59 PM,06/01/2026 01:29:05 AM,NYPD,New York City Police Department,Abandoned Vehicle,With License Plate,NaN,Street/Sidewalk,11219.0,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.641689,-73.994080,POINT (-73.994079777458 40.641688932859)
3,69192183,05/31/2026 11:43:24 PM,06/01/2026 12:35:56 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11210.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.635238,-73.942093,POINT (-73.942093350215 40.635237980126)
4,69188027,05/31/2026 11:43:06 PM,06/01/2026 12:20:18 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11237.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.703853,-73.924383,POINT (-73.924382747932 40.703852624794)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106397,68844679,05/01/2026 12:05:50 AM,05/01/2026 12:38:38 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11206.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.692532,-73.947741,POINT (-73.947741147935 40.692532455153)
106398,68843239,05/01/2026 12:05:05 AM,05/01/2026 12:58:46 AM,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,NaN,Club/Bar/Restaurant,11249.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.712056,-73.962858,POINT (-73.962857645604 40.712056355899)
106399,68841844,05/01/2026 12:03:02 AM,05/01/2026 01:15:02 AM,NYPD,New York City Police Department,Blocked Driveway,No Access,NaN,Street/Sidewalk,11234.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.616508,-73.928426,POINT (-73.928425820245 40.616508065096)
106400,68845662,05/01/2026 12:00:55 AM,05/18/2026 12:00:00 AM,DOB,Department of Buildings,Emergency Response Team (ERT),Lights From Parking Lot Shining On Building,NaN,NaN,11237.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.698725,-73.911658,POINT (-73.911658026634 40.698724571005)


## Step 2 — First look

Before you clean anything, you need to know what you're dealing with.
Three questions to answer in the next cell:

1. How many rows and columns does `df` have? (attribute: `.shape`)
2. What are the column names? (attribute: `.columns`)
3. What do the first 5 rows look like? (method: `.head()`)

In [46]:
import pandas as pd                          # load the tool
df = pd.read_csv('../data/311_Service_Requests_from_2020_to_Present_20260608.csv')   # load the data

df.shape
df.columns
df.head(5)


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,69189924,05/31/2026 11:44:40 PM,06/01/2026 02:08:25 AM,NYPD,New York City Police Department,Illegal Parking,Commercial Overnight Parking,NaN,Street/Sidewalk,11214.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.612992,-74.002993,POINT (-74.00299305542 40.612992270116)
1,69195629,05/31/2026 11:44:28 PM,06/01/2026 02:08:47 AM,NYPD,New York City Police Department,Non-Emergency Police Matter,Trespassing,NaN,Residential Building/House,11212.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.661356,-73.907126,POINT (-73.907126367225 40.66135638569)
2,69188434,05/31/2026 11:43:59 PM,06/01/2026 01:29:05 AM,NYPD,New York City Police Department,Abandoned Vehicle,With License Plate,NaN,Street/Sidewalk,11219.0,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.641689,-73.994080,POINT (-73.994079777458 40.641688932859)
3,69192183,05/31/2026 11:43:24 PM,06/01/2026 12:35:56 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11210.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.635238,-73.942093,POINT (-73.942093350215 40.635237980126)
4,69188027,05/31/2026 11:43:06 PM,06/01/2026 12:20:18 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11237.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.703853,-73.924383,POINT (-73.924382747932 40.703852624794)


## Step 3 — Find the dirt

Two things to check before writing any cleaning code:

1. **Null counts** — how many missing values are in each column?
   Try `df.isnull().sum()`. Which columns are worst?

2. **Unique complaint types** — how many distinct values are in the `Complaint Type` column?
   Try `df['Complaint Type'].value_counts()`. Scroll through it.
   Notice anything odd about the formatting?

In [75]:
#print(df.isnull().sum())
df['Problem (formerly Complaint Type)'].value_counts()


Problem (formerly Complaint Type)
Illegal Parking                21948
Noise - Residential            11402
Blocked Driveway                6017
Noise - Street/Sidewalk         5399
UNSANITARY CONDITION            3578
                               ...  
Animal Facility - No Permit        1
Unsanitary Animal Facility         1
Highway Sign - Damaged             1
Green Taxi Report                  1
Dept of Investigations             1
Name: count, Length: 155, dtype: int64

## Step 4 — Function 1: `clean_complaint_type`

**What it should do:** Take a pandas Series (one column) and return a cleaned version where:
- All values are stripped of leading/trailing whitespace
- All values are in Title Case (e.g. `"HEAT/HOT WATER"` → `"Heat/Hot Water"`)
- Any null values stay null (don't crash on them)

**Math teacher analogy:** This is like standardizing how students write their answers
on a rubric — same content, consistent format.

**Hint when you're stuck:** pandas string methods live under `.str` — for example,
`series.str.strip()` strips whitespace. You can chain them: `series.str.strip().str.title()`.

In [ ]:
def clean_complaint_type(series):
    return series.str.strip(' ').str.title()

sample = df['Problem (formerly Complaint Type)'].head(20)
print(clean_complaint_type(sample))


0                 Illegal Parking
1     Non-Emergency Police Matter
2               Abandoned Vehicle
3             Noise - Residential
4             Noise - Residential
5         Noise - Street/Sidewalk
6         Noise - Street/Sidewalk
7                 Illegal Parking
8              Food Establishment
9         Noise - Street/Sidewalk
10                Illegal Parking
11        Noise - Street/Sidewalk
12            Noise - Residential
13            Noise - Residential
14                Illegal Parking
15             Noise - Commercial
16       Traffic Signal Condition
17        Noise - Street/Sidewalk
18                Illegal Parking
19            Noise - Residential
Name: Problem (formerly Complaint Type), dtype: str


## Step 5 — Function 2: `parse_created_date`

**What it should do:** Take the `Created Date` column (strings like `"03/15/2024 02:00:00 AM"`)  
and return a Series of proper `datetime` objects that pandas understands as dates.

**Why this matters:** As long as dates are strings, you can't filter by month, calculate
how many days between complaints, or plot a timeline. Converting to datetime unlocks all of that.

**Hint when you're stuck:** The function is `pd.to_datetime()`. Pass it a Series.

In [93]:
def parse_created_date(series):
    return pd.to_datetime(series)


# Test it:
sample = df['Created Date'].head(20)
print(parse_created_date(sample))
print(parse_created_date(sample).dtype)  # should say datetime64


0    2026-05-31 23:44:40
1    2026-05-31 23:44:28
2    2026-05-31 23:43:59
3    2026-05-31 23:43:24
4    2026-05-31 23:43:06
5    2026-05-31 23:42:51
6    2026-05-31 23:42:36
7    2026-05-31 23:42:24
8    2026-05-31 23:42:15
9    2026-05-31 23:42:04
10   2026-05-31 23:41:40
11   2026-05-31 23:41:40
12   2026-05-31 23:41:14
13   2026-05-31 23:41:02
14   2026-05-31 23:40:52
15   2026-05-31 23:40:39
16   2026-05-31 23:40:00
17   2026-05-31 23:38:36
18   2026-05-31 23:38:35
19   2026-05-31 23:38:13
Name: Created Date, dtype: datetime64[us]
datetime64[us]


/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_31827/835588796.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series)
/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_31827/835588796.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series)


## Step 6 — Function 3: `flag_missing_location`

**What it should do:** Take the DataFrame `df` and return a boolean Series —  
`True` where the row is missing a location (lat **or** lon is null), `False` otherwise.

**Why this matters:** Later, when we map complaints by neighborhood, rows without
coordinates are useless. This flag lets us filter them out or count them separately.

**Columns to check:** `Latitude` and `Longitude`.

**Hint when you're stuck:** `df['Latitude'].isnull()` returns a boolean Series.
The `|` operator combines two boolean Series with OR.

In [97]:
def flag_missing_location(df):
    return df['Latitude'].isnull() | df['Longitude'].isnull()
    


# Test it:
flags = flag_missing_location(df)
print(flags.sum(), "rows missing location out of", len(df))


1781 rows missing location out of 106402


## Step 7 — Apply all three functions

Now wire the functions into the DataFrame. The pattern is:

```
df['new_column'] = your_function(df['source_column'])
```

Create three new columns:
- `complaint_type_clean` ← result of `clean_complaint_type`
- `created_dt` ← result of `parse_created_date`  
- `missing_location` ← result of `flag_missing_location`

Then print the first 5 rows of just those three new columns to verify they look right.

In [105]:
df['complaint_type_clean'] = clean_complaint_type(df['Problem (formerly Complaint Type)'])
df['created_dt'] = parse_created_date(df['Created Date'])
df['missing_location'] = flag_missing_location(df)

print(df[['missing_location', 'created_dt', 'complaint_type_clean']].head(5))
df.to_csv('../data/311_cleaned.csv', index=False)


/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_31827/835588796.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series)


   missing_location          created_dt         complaint_type_clean
0             False 2026-05-31 23:44:40              Illegal Parking
1             False 2026-05-31 23:44:28  Non-Emergency Police Matter
2             False 2026-05-31 23:43:59            Abandoned Vehicle
3             False 2026-05-31 23:43:24          Noise - Residential
4             False 2026-05-31 23:43:06          Noise - Residential


## Day 2 Recap

| What you did | Why it matters |
|---|---|
| `.isnull().sum()` | Found which columns have gaps before trusting the data |
| `.str.strip().str.title()` | Standardized free-text so groupby/counts work correctly |
| `pd.to_datetime()` | Unlocked time-based filtering and math |
| Boolean flag column | Lets you filter or audit bad rows without losing them |

### What's coming on Day 3

We'll use the clean `complaint_type_clean` and `created_dt` columns to answer a real question:
**Which Bushwick complaint types spike in winter?**  
That means groupby, aggregation, and your first pandas plot.